# Etapa 1 — Ambiente

Objetivo: confirmar que o ambiente `cambio` tem tudo o que o projeto precisa e que conseguimos **conversar com as três fontes de dados**:

1. **SGS do Banco Central** (PTAX, Selic) — via `python-bcb`
2. **Pesquisa Focus** (expectativas) — também via `python-bcb`, mas por outra API (OData)
3. **FRED** (Treasury 10a, VIX, dólar global) — via `fredapi`, que exige chave

Nada é salvo aqui: é só um teste de "encanamento". A coleta de verdade é a etapa 2.

In [1]:
import os
import sys

import pandas as pd
import statsmodels
from bcb import sgs, Expectativas  # SGS = séries temporais do BCB; Expectativas = Focus
from fredapi import Fred

# Confere se o notebook está rodando no ambiente certo (deve terminar em envs\cambio\python.exe)
print("Python:", sys.executable)
print("pandas:", pd.__version__, "| statsmodels:", statsmodels.__version__)

Python: C:\Users\guire\anaconda3\envs\cambio\python.exe
pandas: 3.1.0 | statsmodels: 0.15.0


## 1. SGS — PTAX de venda (série 1) e meta Selic (série 432)

O SGS identifica cada série por um número. `sgs.get` recebe um dicionário `{nome_da_coluna: código}` e devolve um DataFrame indexado por data. `last=5` pede só as 5 últimas observações.

In [2]:
teste_sgs = sgs.get({"ptax_venda": 1, "selic_meta": 432}, last=5)
teste_sgs

,ptax_venda,selic_meta
Date,,
2026-10-30,5.4212,NaN
2026-11-03,5.4935,NaN
2026-11-04,5.4778,NaN
2026-11-05,5.4621,NaN
2026-11-06,5.5013,NaN
2026-12-05,NaN,13.75
2026-12-06,NaN,13.75
2026-12-07,NaN,13.75
2026-12-08,NaN,13.75


## 2. Focus — expectativas anuais

O Focus é uma API diferente (OData): primeiro escolhemos um *endpoint* (uma tabela) e depois montamos a consulta. Aqui só pegamos algumas linhas para ver quais indicadores existem — na etapa 2 vamos filtrar os fiscais (resultado primário, dívida líquida).

In [3]:
em = Expectativas()
ep = em.get_endpoint("ExpectativasMercadoAnuais")

# Pega as observações mais recentes e lista os indicadores disponíveis
teste_focus = ep.query().filter(ep.Data >= "2026-10-01").limit(2000).collect()
print(teste_focus["Indicador"].unique())

<StringArray>
[              'Balança comercial',                          'Câmbio',
                  'Conta corrente',   'Dívida bruta do governo geral',
 'Dívida líquida do setor público',                           'IGP-M']
Length: 6, dtype: str


## 3. FRED — Treasury 10 anos, VIX e dólar amplo

A chave é lida da variável de ambiente `FRED_API_KEY` — ela **nunca** aparece escrita no código.

In [4]:
fred = Fred(api_key=os.environ["FRED_API_KEY"])

series_fred = {"ust10": "DGS10", "vix": "VIXCLS", "dolar_amplo": "DTWEXBGS"}
teste_fred = pd.DataFrame(
    {nome: fred.get_series(codigo, observation_start="2026-10-01") for nome, codigo in series_fred.items()}
)
teste_fred.tail()

,ust10,vix,dolar_amplo
2026-10-30,4.94,19.42,121.6656
2026-11-02,4.91,21.15,NaN
2026-11-03,4.95,20.83,NaN
2026-11-04,4.90,16.61,NaN
2026-11-05,4.84,28.18,NaN


## O que conferir

- **Python** termina em `envs\cambio\python.exe` (senão, troque o kernel).
- **PTAX** perto do câmbio que você vê no noticiário; **Selic** igual à meta vigente do Copom.
- **Focus** lista indicadores como IPCA, Selic, Câmbio e os fiscais.
- **FRED**: Treasury em % a.a. (ex.: 4,xx), VIX em pontos (tipicamente 12–30). Note os `NaN`: feriados americanos e o fato de o DTWEXBGS ser publicado com defasagem — vamos tratar isso na etapa 3.